# Tibeb AI — GPU Training (v2)
**Data auto-downloads from HuggingFace — no manual upload needed!**

Runtime → Change runtime type → **A100 GPU** (or T4 for free tier)

In [ ]:
# Check GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Install dependencies
!pip install -q "transformers>=4.44.0" "datasets>=2.18.0" "peft>=0.10.0" "trl>=0.8.0" \
  "accelerate>=0.28.0" "bitsandbytes>=0.43.0" safetensors huggingface_hub
!pip install -q flash-attn --no-build-isolation 2>/dev/null || echo "flash-attn skipped"

In [ ]:
# Download data from HuggingFace
import os, json
from huggingface_hub import hf_hub_download

DATA_PATH = '/content/tibeb_unified_train.jsonl'
if not os.path.exists(DATA_PATH):
    print('Downloading training data from HuggingFace...')
    hf_hub_download(
        repo_id='nahommohan/tibeb-training-data',
        filename='data/tibeb_unified_train.jsonl',
        repo_type='dataset',
        local_dir='/content',
    )
    DATA_PATH = '/content/data/tibeb_unified_train.jsonl'
    print(f'Downloaded: {DATA_PATH}')
else:
    print(f'Found: {DATA_PATH}')

!wc -l "{DATA_PATH}"

In [ ]:
# Load and format data
SYSTEM_PROMPT = (
    "ቲብብ ነኝ — የኢትዮጵያ የፋይናንስ AI ረዳት።"
    " ስለ ኢንቨስትመንት፣ ቁጠባ፣ አክሲዮን ገበያ፣ ቲ-ቢል፣ እና የገንዘብ ጉዳዮች በአማርኛ እረዳለሁ።"
    " ሁልጊዜ አክብሮት ያለው፣ ትምህርታዊ፣ እና ግልጽ መልስ እሰጣለሁ።"
    " ቀጥተኛ የኢንቨስትመንት ምክር አልሰጥም — ተገቢውን ባለሙያ እንዲያማክሩ እመክራለሁ።"
)

FINANCIAL = {'tibeb_financial', 'tibeb_financial_v2', 'sujet_finance',
             'tibeb_safety', 'tibeb_voice', 'tibeb_deep'}

import random
random.seed(42)

rows = []
skipped = 0
with open(DATA_PATH, encoding='utf-8') as f:
    for line in f:
        r = json.loads(line.strip())
        if not r.get('instruction') or not r.get('output'): skipped += 1; continue
        out = r['output'].strip()
        if len(out) < 10: skipped += 1; continue
        inp = (r.get('input') or '').strip()
        if inp and out == inp: skipped += 1; continue

        user_msg = f"{r['instruction']}\n\n{inp}" if inp else r['instruction']
        msgs = []
        if r.get('source', '') in FINANCIAL:
            msgs.append({'role': 'system', 'content': SYSTEM_PROMPT})
        msgs.append({'role': 'user', 'content': user_msg})
        msgs.append({'role': 'assistant', 'content': out})
        rows.append({'messages': msgs})

random.shuffle(rows)
split = int(len(rows) * 0.98)
train_data, eval_data = rows[:split], rows[split:]
print(f'Train: {len(train_data):,} | Eval: {len(eval_data):,} | Skipped: {skipped:,}')

In [ ]:
# Load model (4-bit QLoRA)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import Dataset

MODEL_ID = 'CohereForAI/aya-expanse-8b'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True),
    device_map='auto', torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model)

model = get_peft_model(model, LoraConfig(
    r=64, lora_alpha=128, lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none', task_type='CAUSAL_LM',
))
model.print_trainable_parameters()

In [ ]:
# Train
OUTPUT_DIR = '/content/tibeb-sft-gpu'

def fmt(example):
    return tokenizer.apply_chat_template(example['messages'], tokenize=False, add_generation_prompt=False)

trainer = SFTTrainer(
    model=model,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=8,
        learning_rate=2e-5,
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        weight_decay=0.01,
        bf16=True,
        logging_steps=25,
        save_strategy='steps',
        save_steps=500,
        eval_strategy='steps',
        eval_steps=500,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        max_grad_norm=1.0,
        report_to='none',
        seed=42,
    ),
    train_dataset=Dataset.from_list(train_data),
    eval_dataset=Dataset.from_list(eval_data),
    formatting_func=fmt,
    max_seq_length=1024,
    packing=True,
)

print('Starting training...')
trainer.train()
print('Done!')

In [ ]:
# Save & push to HuggingFace
FINAL = '/content/tibeb-sft-final'
trainer.save_model(FINAL)
tokenizer.save_pretrained(FINAL)
print(f'Saved locally: {FINAL}')

# Push to HuggingFace
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path=FINAL,
    repo_id='nahommohan/tibeb-sft-adapter',
    path_in_repo='gpu-adapter-v1',
    repo_type='model',
)
print('Pushed to HuggingFace: nahommohan/tibeb-sft-adapter/gpu-adapter-v1')

In [ ]:
# Test it
model.eval()
prompts = [
    'ቲ-ቢል ምንድነው?',
    'ለጀማሪ ኢንቨስተር ምን ትመክራለህ?',
    'ቁጠባ ባንክ ውስጥ ማስቀመጥ ወይስ ኢንቨስት ማድረግ ይሻላል?',
    'የኢትዮጵያ ሴኩሪቲስ ኤክስቼንጅ ምንድነው?',
    'ሰላም! ስለ ኢትዮጵያ የካፒታል ገበያ ንገረኝ።',
]

for p in prompts:
    msgs = [{'role':'system','content':SYSTEM_PROMPT}, {'role':'user','content':p}]
    inp = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tokenizer(inp, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=300, temperature=0.7,
                             top_p=0.9, do_sample=True, repetition_penalty=1.1)
    resp = tokenizer.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'\n{"─"*60}\n  Q: {p}\n{"─"*60}\n  {resp.strip()[:400]}\n')